# 19 — Performance, callbacks, and code generation

**Objectives.** Measure graph construction separately from evaluation, use mapped evaluation, generate C, and understand extension boundaries. Optimize only after profiling representative dimensions. SX often evaluates small scalar formulas efficiently; MX prevents large matrix algorithms from being scalar-expanded.


In [ ]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

### Map a dynamics graph and generate standalone C code

The cell defines a four-state MX dynamics expression and its exact state
Jacobian. `Function.map(100)` evaluates one graph independently across 100
columns, avoiding Python-loop overhead. `Function.generate` emits C source
and a header for the same graph in the ignored `generated/` directory.

Elementwise, the dynamics are

$$
f_i(x,p)=\sin(x_i)+p_i x_i^2,
$$

with diagonal Jacobian

$$
\frac{\partial f_i}{\partial x_j}=
\begin{cases}
\cos(x_i)+2p_i x_i,&i=j,\\
0,&i\ne j.
\end{cases}
$$

Mapping and code generation alter how the computation is executed, not
the mathematical function being evaluated.


In [ ]:
import os
from pathlib import Path

import casadi as ca

x = ca.MX.sym("x", 4)
parameters = ca.MX.sym("parameters", 4)
dynamics = ca.sin(x) + parameters * x**2
dynamics_function = ca.Function(
    "generated_dynamics",
    [x, parameters],
    [dynamics, ca.jacobian(dynamics, x)],
    ["x", "parameters"],
    ["dynamics", "state_jacobian"],
)

mapped_dynamics = dynamics_function.map(100)
mapped_result = mapped_dynamics(
    ca.DM.zeros(4, 100),
    ca.DM.ones(4, 100),
)
print(
    "mapped output shape:",
    mapped_result[0].shape,
    "function nodes:",
    dynamics_function.n_nodes(),
)

output_directory = Path("generated").resolve()
output_directory.mkdir(exist_ok=True)
original_directory = Path.cwd()
try:
    os.chdir(output_directory)
    dynamics_function.generate(
        "generated_dynamics.c",
        {"with_header": True},
    )
finally:
    os.chdir(original_directory)

assert (output_directory / "generated_dynamics.c").exists()

### Wrap external Python logic with the Callback API

A `ca.Callback` subclass declares its input/output counts and implements
numerical evaluation. `construct` registers the callback with CasADi;
`enable_fd=True` tells CasADi to approximate derivatives by finite
differences because no analytic derivative callback is supplied. The
callback is then embedded in an MX `Function` and differentiated with
`ca.jacobian`.

The external operation is

$$
y(z)=z^2,\qquad \frac{dy}{dz}=2z.
$$

At $z=3$, the expected value and derivative are $9$ and $6$.
This pattern is useful for legacy or third-party numerical code, although
native symbolic expressions are preferable when they are available.


In [ ]:
class SquareCallback(ca.Callback):
    """Expose a Python square operation through CasADi's Callback API."""

    def __init__(self, name="square_callback"):
        ca.Callback.__init__(self)

        # enable_fd lets CasADi finite-difference this external operation.
        self.construct(name, {"enable_fd": True})

    def get_n_in(self):
        """Declare the number of Callback inputs."""
        return 1

    def get_n_out(self):
        """Declare the number of Callback outputs."""
        return 1

    def eval(self, arguments):
        """Evaluate the external operation numerically."""
        return [arguments[0] ** 2]


square_callback = SquareCallback()
argument = ca.MX.sym("argument")
wrapped_callback = ca.Function(
    "wrapped_callback",
    [argument],
    [square_callback(argument)],
)
callback_derivative = ca.Function(
    "callback_derivative",
    [argument],
    [ca.jacobian(wrapped_callback(argument), argument)],
)
print(
    "callback value:",
    wrapped_callback(3),
    "finite-difference derivative:",
    callback_derivative(3),
)

Generated C removes virtual-machine overhead and can run without CasADi when linked against its self-contained API. `external` loads compatible compiled functions; `Function.jit` compiles C source at runtime. `Callback` connects Python/external logic but Python callbacks are slower and need supplied derivatives or finite differences.

**Common mistakes:** timing graph construction as evaluation, generating one enormous expanded function, and assuming threads make a non-thread-safe callback safe. **Recap:** establish graph boundaries, map repeated work, record timings, then compare VM, expanded, and generated implementations.
